# R², Adjusted R² and MAPE Lab

Explore the mathematics, identities, and behaviors of relative and percentage error metrics: the three sums of squares, the conditional nature of the SST = SSR + SSE identity, negative R², Adjusted R² feature-count penalties, and percentage error variants (MAPE, sMAPE, WAPE).

In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

## Helper Functions

Let's define a helper to calculate Adjusted $R^2$ from the standard formula.

In [ ]:
def adjusted_r2(r2_value, n_rows, n_features):
    """1 - (1 - R2) * (n - 1) / (n - p - 1). nan when p >= n - 1."""
    dof = n_rows - n_features - 1
    if dof <= 0:
        return float("nan")
    return 1 - (1 - r2_value) * (n_rows - 1) / dof

## 1. The Three Sums of Squares

Let's compute $SST$, $SSE$, and $SSR$ on five houses and verify that $SST = SSR + SSE + 2 \cdot \text{cross}$ holds exactly, but the cross term is only zero when the predictions are derived from OLS.

In [ ]:
y = np.array([52.0, 60.0, 71.0, 88.0, 95.0])
y_hat = np.array([49.0, 61.0, 69.0, 92.0, 95.0])
r = y - y_hat
y_bar = y.mean()

sst = np.sum((y - y_bar) ** 2)
sse = np.sum(r ** 2)
ssr = np.sum((y_hat - y_bar) ** 2)
cross = np.sum((y_hat - y_bar) * r)

print("1. SST, SSE, SSR ON HAND-PICKED PREDICTIONS")
print(f"   y = {y}   y_bar = {y_bar}")
print(f"   SST = {sst:.4f}   SSE = {sse:.4f}   SSR = {ssr:.4f}")
print(f"   R2 = 1 - SSE/SST     = {1 - sse / sst:.8f}")
print(f"   r2_score(y, y_hat)   = {r2_score(y, y_hat):.8f}")
print(f"   SSR/SST              = {ssr / sst:.8f}   <-- above 1")
print(f"   SSR + SSE            = {ssr + sse:.4f}  vs  SST = {sst:.4f}")
print(f"   cross = sum((y_hat - y_bar) * r) = {cross:.4f}")
print(f"   SSR + SSE + 2*cross  = {ssr + sse + 2 * cross:.4f}   <-- now equals SST")

Now let's fit the line properly via OLS and watch the cross term vanish algebraically.

In [ ]:
print("\n2. THE SAME Y, FITTED PROPERLY")
X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
fit = LinearRegression().fit(X, y)
y_fit = fit.predict(X)
r_fit = y - y_fit
sse_fit = np.sum(r_fit ** 2)
ssr_fit = np.sum((y_fit - y_bar) ** 2)

print(f"   b0 = {fit.intercept_:.4f}   b1 = {fit.coef_[0]:.4f}")
print(f"   y_fit = {y_fit}   residuals = {r_fit}")
print(f"   sum(residuals) = {r_fit.sum():.2e}")
print(f"   SSE = {sse_fit:.4f}   SSR = {ssr_fit:.4f}   SST = {sst:.4f}")
print(f"   SSR + SSE = {ssr_fit + sse_fit:.4f}   <-- equals SST")
print(f"   cross term = {np.sum((y_fit - y_bar) * r_fit):.2e}   <-- zero")
print(f"   1 - SSE/SST = {1 - sse_fit / sst:.8f}   r2_score = {r2_score(y, y_fit):.8f}")
pearson = np.corrcoef(X.ravel(), y)[0, 1]
print(f"   corr(x, y)^2 = {pearson ** 2:.8f}  <-- equals R2 for simple regression")

## 2. Zero and Negative $R^2$

Observe how predicting a bad constant like $60$ yields a negative $R^2 = -0.6586$. Check why $R^2$ is asymmetric in its arguments while MSE is symmetric.

In [ ]:
print("3. ZERO AND NEGATIVE R2")
print(f"   predict y_bar ({y_bar}) everywhere: R2 = {r2_score(y, np.full_like(y, y_bar)):.8f}")
flat_60 = np.full_like(y, 60.0)
sse_60 = np.sum((y - flat_60) ** 2)
print(f"   predict 60 everywhere: SSE = {sse_60:.1f}, R2 = {r2_score(y, flat_60):.8f}")
print(f"   r2_score(y, y_fit)  (correct) = {r2_score(y, y_fit):.8f}")
print(f"   r2_score(y_fit, y)  (swapped) = {r2_score(y_fit, y):.8f}  <-- asymmetric!")

## 3. Adding Junk Features

Let's add 12 columns of pure Gaussian noise on $n_t = 40$ train rows. Watch in-sample $R^2$ rise monotonically, while Adjusted $R^2$ and out-of-sample test $R^2$ fall.

In [ ]:
print("4. TWELVE JUNK FEATURES ADDED ONE AT A TIME")
n, n_train = 60, 40
x_real = np.random.uniform(0, 10, size=n)
target = 5.0 + 3.0 * x_real + np.random.normal(0, 2.0, size=n)
design = np.column_stack([x_real, np.random.normal(0, 1.0, size=(n, 12))])

print(f"   {'p':>3} {'train R2':>11} {'adj R2':>11} {'test R2':>11}")
for p in range(1, 14):
    xs = design[:, :p]
    model = LinearRegression().fit(xs[:n_train], target[:n_train])
    tr = r2_score(target[:n_train], model.predict(xs[:n_train]))
    te = r2_score(target[n_train:], model.predict(xs[n_train:]))
    print(f"   {p:>3} {tr:>11.6f} {adjusted_r2(tr, n_train, p):>11.6f} {te:>11.6f}")

print("\n   Enough features to interpolate (y is pure noise, unrelated to X):")
m = 8
xs_small = np.random.normal(0, 1.0, size=(m, m - 1))
ys_small = np.random.normal(100, 15, size=m)
interp = LinearRegression().fit(xs_small, ys_small)
r2_interp = r2_score(ys_small, interp.predict(xs_small))
print(f"     n = {m}, p = {m - 1}: in-sample R2 = {r2_interp:.10f}")
print(f"     adjusted R2 = {adjusted_r2(r2_interp, m, m - 1)}")

## 4. Model A versus Model B

Compare Model A ($p=2$) against Model B ($p=7$) on 50 rows, demonstrating how degrees of freedom are consumed.

In [ ]:
print("5. TWO CANDIDATE MODELS, 50 ROWS")
rows = 50
for name, p_feat, r2_val in [("Model A", 2, 0.820), ("Model B", 7, 0.835)]:
    print(f"   {name}: p = {p_feat}, R2 = {r2_val:.3f}, adjusted R2 = {adjusted_r2(r2_val, rows, p_feat):.8f}")
target_adj = adjusted_r2(0.820, rows, 2)
needed = 1 - (1 - target_adj) * (rows - 8) / (rows - 1)
print(f"   Model B must reach R2 >= {needed:.8f} to tie Model A's adjusted R2")

## 5. Percentage Error Pathologies

Let's check APE, MAPE, sMAPE, and WAPE on our 5-house residuals, and look at zero values and over-prediction asymmetries.

In [ ]:
print("6. PERCENTAGE ERROR PATHOLOGIES")
ape = np.abs(r) / np.abs(y)
mae = mean_absolute_error(y, y_hat)
sk_mape = mean_absolute_percentage_error(y, y_hat)

print(f"   per-row APE (%)   = {ape * 100}")
print(f"   MAPE by hand      = {ape.mean() * 100:.6f} %")
print(f"   sklearn MAPE      = {sk_mape:.8f}  <-- fraction! x100 = {sk_mape * 100:.6f} %")
print(f"   sMAPE             = "
      f"{(np.abs(r) / ((np.abs(y) + np.abs(y_hat)) / 2)).mean() * 100:.6f} %")
print(f"   WAPE              = {np.abs(r).sum() / np.abs(y).sum() * 100:.6f} %")
print(f"   MAE / y_bar       = {mae / y_bar * 100:.6f} %  <-- identical to WAPE")

print("\n   A zero in y true:")
y_zero, p_zero = np.array([0.0, 60.0, 71.0]), np.array([1.0, 61.0, 69.0])
print(f"     sklearn MAPE = {mean_absolute_percentage_error(y_zero, p_zero):.6e}")
print(f"     WAPE         = {np.abs(y_zero - p_zero).sum() / np.abs(y_zero).sum() * 100:.6f} %")

print("\n   Same MAE, MAPE 100x apart:")
pair = np.array([10.0, 1000.0])
for const in (10.0, 1000.0):
    pred_const = np.full_like(pair, const)
    print(f"     predict {const:>6.0f}: MAE = {mean_absolute_error(pair, pred_const):7.1f}, "
          f"MAPE = {mean_absolute_percentage_error(pair, pred_const) * 100:10.1f} %")

print("\n   Unbounded above, capped at 100% below (y = 100):")
for pv in (0.0, 50.0, 150.0, 300.0, 10000.0):
    print(f"     y_hat = {pv:>8.0f}  ->  APE = {abs(100 - pv):>8.1f} %")